In [1]:
%run ../../Latex_macros.ipynb

<IPython.core.display.Latex object>

# Policy gradient II: what weights each step?

## Where this session fits

- **Series goal**
    - choose good actions by trial and error
    - from small tables to robots and language models
- **Position: Part B (policy gradient), session 2 of its 4 sessions**
    - session 1 gave the theorem: $\nabla_\theta J$ is an average of $\nabla_\theta \log \pi_\theta$ times a **weight**; the raw return was the weight
    - **the limitation we start from:** the raw return is a very noisy weight, so the estimate is very noisy (high variance)
    - this session fills slot 1, and part of slot 3 (where the baseline comes from): *what should weight each step?*
- **The question:** what can weight each step instead of the raw return, without changing the expected gradient, or at a price we can state?
- **By the end you can**
    - say where the noise in the weight comes from
    - define a baseline and the advantage, and say why a baseline that does not depend on the action does not bias the gradient
    - write the advantage form of the gradient and read REINFORCE as one choice of advantage
    - say what a critic is, how it supplies a baseline, and where its bias comes from (the bootstrap, not the baseline)
    - place REINFORCE and actor-critic in the slot table


**Where this session sits in Part B**

<img src="images/partB_arc_b2.png" width=95%>


# Variance returns: where the noise in the weight comes from

**Why this section?** Bias and variance were defined for the *targets* of Monte Carlo and TD. The same two words now describe the *weight* in the policy gradient.

- recall (from sampling)
    - **bias:** how far the average of an estimate is from the truth
    - **variance:** how much the estimate fluctuates from sample to sample
- the weight in the theorem, a sampled return, is **unbiased** (session 1) but has **high variance**, for three reasons
    - **the whole return is in it:** every action gets the full return, including the part that every action would have received anyway
    - **it is one sample:** a single sampled episode, whose rewards depend on the environment's randomness and on the policy's own sampling
    - **no per-step credit:** the same number weights every step, so with sparse rewards we cannot say *which* step deserves the credit
- each reason has a fix, and the fixes are this session
    - subtract something every action shares: a **baseline**
    - give each step its own weight: **credit assignment**, the **advantage** (using only the rewards that follow a step, or a critic)
    - replace part of the sampled return by an estimate: a **critic** (it lowers variance, at the price of bias, as TD did; it also helps with credit)


## A baseline: subtract what every action would get anyway

**Why this section?** The first reason: the weight contains a part shared by all actions. Remove it and the signal that remains is what the action added.

- suppose every return is positive
    - the theorem pushes *up* every sampled action, in proportion to the raw return, even though most of that return would have come anyway
- a **baseline** $b(\state)$ is a number that depends on the state but **not on the action**, subtracted from the weight
    - for example, the average return from that state
- the **advantage** is the return relative to the baseline
    - $\advseq = \rlret - b(\state)$: positive if the action did better than usual, negative if worse
- the *expected* gradient is the same whether or not $b$ is subtracted; only the spread changes

- figure: the same estimator with and without a baseline; the spread falls by a factor of about $2.4$ ($1.16$ to $0.49$); the two bumps in the left panel are the two signs of the score (action $0$ sampled or not)

<img src="images/baseline_variance.png" width=95%>


**Why a baseline does not bias the gradient**

- the score function $\nabla_\theta \log \pi_\theta$ has zero mean (shown in the appendix to session 1)

$$
\E_{\actseq \sim \pi_\theta}\bigl[ \nabla_\theta \log \pi_\theta(\actseq \mid \state) \, b(\state) \bigr] = b(\state) \, \E\bigl[ \nabla_\theta \log \pi_\theta(\actseq \mid \state) \bigr] = 0
$$

- this needs $b$ **not to depend on the action**: then it comes out of the expectation
- so subtracting $b$ leaves the expected gradient unchanged; what changes is the variance
- a good baseline cancels the part of the return that all actions share
- the exact advantage is $\advseq = \actvalfun_\pi(\state, \act) - \statevalfun_\pi(\state)$
    - because $\E[\rlret_\tt \mid \state, \act] = \actvalfun_\pi(\state, \act)$, the sampled $\rlret_\tt - b$ is a sample of $\actvalfun_\pi - b$: of the exact advantage when $b = \statevalfun_\pi(\state)$
- caution: a baseline computed from the same sample (for example a group mean that includes it) depends slightly on the action; the leave-one-out mean avoids this


## Credit assignment: one weight per step

**Why this section?** The second reason: one weight for a whole episode cannot say which step deserves the credit.

- the episode-return form gives *every* step the same weight
    - a language model writes a 100-word masterpiece and mistypes the last character: the final reward is low, and *all* steps are punished though one step is to blame
    - with sparse rewards we cannot tell good steps from bad ones, so learning is slow (high variance)
- the fix: an **advantage per step**, $\advseq_{\tau,\tt}$, which sets the force and direction of each step's gradient
    - two mechanisms give per-step credit: use only the rewards that *follow* a step (the return-to-go $\rlret_{\tt,\tau}$ instead of $\rlret_{0,\tau}$), and a critic that scores each step
    - a baseline $b(\state)$ alone reduces noise but does not assign credit

| Advantage | Effect on the gradient | Result |
|:--|:--|:--|
| **positive** ($\advseq_{\tau,\tt} > 0$) | positive multiplier | **accelerate**: better than usual, make it more likely |
| **negative** ($\advseq_{\tau,\tt} < 0$) | flips the sign | **brake**: below expectation, make it less likely |
| **about zero** | nullifies the gradient | **maintain**: as expected, nothing to learn |

- advantage-based methods are the basis of modern policy gradient; one well-known early example is A3C (Mnih et al., 2016)


## The unified form: slot 1 is the advantage

**Why this section?** With a weight per step, the methods differ only in how they compute it. One formula makes them comparable.

$$
\nabla_\theta J(\theta) = \E_{\tau \sim \pi_\theta} \left[ \sum_{\tt=0}^{|\tau|-1} \nabla_\theta \log \pi_\theta(\actseq_{\tau,\tt} \mid \stateseq_{\tau,\tt}) \; \advseq_{\tau,\tt} \right]
$$

- the weight of session 1's slot 1 is now $\advseq_{\tau,\tt}$; the methods of this part are variations of this "vanilla" gradient
- as in most implementations, the factor $\disc^\tt$ of the exact discounted gradient is dropped (it is $1$ when $\disc = 1$); this is the common practical form, not exactly the discounted gradient
- what remains is to choose $\advseq_{\tau,\tt}$:
    - **REINFORCE:** the return, minus a baseline from past episodes
    - **actor-critic:** the advantage from a *learned* value function
    - (later sessions: PPO uses a critic, GRPO a group of episodes)


# REINFORCE: return minus a moving average

**Why this section?** The simplest method with an advantage, and the reference point for what follows.

- REINFORCE is very close to the theorem: the weight is the return minus a baseline
    - rewards at every step: $\advseq_{\tau,\tt} = \rlret_{\tt,\tau} - b_\tt$, where $b_\tt$ is often a stand-in for $\statevalfun_\pi(\stateseq_\tt)$ (per-step credit through the return-to-go)
    - a single reward at the end: $\advseq_{\tau,\tt} = \rlret_{0,\tau} - b$ for *every* step: the same weight everywhere, so no per-step credit; that needs a critic
- a common baseline for a single reward: a **moving average** of the returns of all episodes so far
    - $b_{\text{new}} = (1 - w)\, b_{\text{old}} + w \, \rlret_{0,\tau}$, with a small weight $w$ (not the learning rate $\alpha$)
    - the baseline used for an episode is its value *before* that episode updates it
- it needs **no critic**: nothing is learned except the policy
- "Monte Carlo" here is the same sense as in sampling: the full return of a complete sampled episode

- for each training prompt $\x$
    - generate an output $\y$ from the policy $\pi_\theta$; its reward $\rew$ comes from an evaluator
    - the baseline is the current moving average $b$ (a constant for this episode)
    - gradient step on $\theta$ in the direction $(\rew - b)\, \nabla_\theta \log \pi_\theta(\y \mid \x)$; then update $b$ by the rule above


**REINFORCE is still noisy**

- the baseline reduced the first reason (the shared part), though a moving average that ignores the state removes only part of it
- the weight is still $\rlret_{\tt,\tau}$ from **one sampled episode**, so it fluctuates with the environment's random responses and the policy's own sampling
- **practical remedy:** several episodes for the *same* prompt, so averaging lowers the variance, and their mean return is a prompt-specific baseline
    - this is the idea behind GRPO (session 4)
- to attack this reason directly we replace part of the sampled return by an *estimate*: a critic


# Actor-critic: a critic and a bootstrap

**Why this section?** The third fix: replace the tail of the sampled return by a learned estimate, exactly as TD replaced the tail of the Monte Carlo return.

- two networks
    - the **actor** $\pi_\theta$: the policy network of Part B
    - the **critic** $\statevalfun_\phi$: a network of parameters $\phi$ that learns state values, as in Part A (a *use* of the value-based machinery, not a return to value-based control)
- the weight is the **TD error** of the critic

$$
\delta_\tt = \rewseq_{\tt+1} + \disc \, \statevalfun_\phi(\stateseq_{\tt+1}) - \statevalfun_\phi(\stateseq_\tt)
\qquad \advseq_\tt \approx \delta_\tt
$$

- $\delta_\tt$ has two parts: a **baseline** ($-\statevalfun_\phi(\stateseq_\tt)$) and a **bootstrap** ($\rewseq_{\tt+1} + \disc \statevalfun_\phi(\stateseq_{\tt+1})$ replacing the rest of the sampled return)
- it is positive when the step went better than the critic predicted: a one-step advantage, a *sample* of the advantage if the critic is right
- the critic is trained by regression toward its own bootstrapped target (a semi-gradient step: the target is treated as a constant)
- as on the unified-form slide, the factor $\disc^\tt$ is dropped
- both are updated from the same sampled steps

- sample $\act \sim \pi_\theta(\cdot \mid \state)$; observe $\rew$ and $\state'$
- $\delta \leftarrow \rew + \disc\,\statevalfun_\phi(\state') - \statevalfun_\phi(\state)$ (the critic's surprise; $\statevalfun_\phi(\text{terminal}) = 0$)
- critic: $\phi \leftarrow \phi + \beta\,\delta\,\nabla_\phi \statevalfun_\phi(\state)$ (reduce its error)
- actor: $\theta \leftarrow \theta + \alpha\,\delta\,\nabla_\theta \log \pi_\theta(\act \mid \state)$ (weight by the surprise)


**What the critic buys and what it costs**

- **buys:** lower variance than a full sampled return: $\delta_\tt$ contains *one* reward and one value, not the whole tail of the episode (as TD(0) against Monte Carlo)
- **costs:** **bias**: if the critic is wrong, $\delta_\tt$ is a biased advantage (the same bootstrapping bias as TD, defined in sampling)
- **costs:** a second network to train, and the two learn together: the critic's target $\statevalfun_\pi$ drifts as the actor changes
- still **on-policy**: the steps come from the current actor


## The estimator dial: from the return to the TD error

**Why this section?** The advantage can be sampled for $\tt$ steps and then closed with the critic. The number of steps is the same *depth* dial as in sampling.

| Advantage weight $\advseq_\tt$ | Steps of sampled reward | Bias | Variance |
|:--|:--|:--|:--|
| return-to-go, no baseline | to the end | none | highest |
| return-to-go minus $b(\state)$ | to the end | none | lower |
| **$p$-step**: $p$ rewards, then the critic | $p$ | critic error, discounted | grows with $p$ |
| TD error $\delta_\tt$ | 1 | critic error | lowest |

- the two ends of the $p$-step rows: the full return minus a baseline $b(\state)$ (a critic can supply $b$) and the TD error
- the first two rows use the full sampled return (the second subtracts a baseline, which a critic may supply and which costs no bias); only the $p$-step rows and the TD error bootstrap from the critic and carry its bias
- ("sampled for $p$ steps" above means $p$ rewards from the episode, then the critic)
- **GAE** (generalized advantage estimation) averages *all* the $p$-step advantages with weights set by a number $\lambda$
    - $\lambda = 0$ gives the TD error, $\lambda = 1$ the full return minus a baseline
    - PPO uses it; it is a one-off tool, so it is in an **appendix** (`RL_B2_Appendix_GAE.ipynb`), worth reading before PPO
- the dial trades bias for variance, the same trade as in sampling; a good setting is found by experiment


## The slot table, first rows

**Why this section?** Session 1 previewed three slots. Here are the first methods placed in them.

| Method | Slot 1: weight $\advseq$ | Slot 2: how far | Slot 3: baseline source (inside slot 1) |
|:--|:--|:--|:--|
| raw theorem (session 1) | return, no baseline | step size $\alpha$ | none |
| **REINFORCE** | return minus baseline | step size $\alpha$ | moving average of returns |
| **actor-critic** | TD error (or a $p$-step advantage, GAE) | step size $\alpha$ | learned critic $\statevalfun_\phi$ |
| PPO | advantage from a critic (GAE) | to be filled (session 3) | learned critic |
| GRPO | group-relative advantage | to be filled (session 4) | mean of a group of episodes |

- the last two rows are previews to be confirmed in sessions 3 and 4
- "step size $\alpha$" in slot 2 is the plain gradient step; session 3 asks how far one update may safely go


## Aside: theoretical versus practical REINFORCE

| Feature | Theoretical REINFORCE | Practical (batch, GRPO-like) |
|:--|:--|:--|
| rollouts per prompt | 1 | several (for example $8$) |
| baseline | global moving average (historical) | group mean (immediate) |
| variance | very high (small learning rates are usual) | reduced by sampling more |
| update frequency | every trajectory | after a group of trajectories |
| compute per update | low | high (heavy generation) |
| analogy | your lifetime golf average | ten swings from the same spot |


# What we established, and what it sets up

- **Established**
    - the raw return is an unbiased but very noisy weight: it contains the shared part, it is one sample, and it gives all steps the same credit
    - a baseline that does not depend on the action leaves the expected gradient unchanged and reduces the variance (the zero-mean fact)
    - the **advantage** (return relative to a baseline) weights each step; the unified form is $\E[\sum \nabla_\theta \log \pi_\theta \; \advseq]$
    - **REINFORCE** uses a moving-average baseline and no critic; it is still noisy
    - **actor-critic** uses a learned critic: the TD error as the advantage, with lower variance and some bias
    - the number of sampled steps before the critic is a dial between the two
- **Sets up**
    - **the limitation:** the weight is better but still noisy, so each step is noisy; yet every batch of episodes is used for one update and then discarded, and a big step on a noisy estimate can ruin the policy
    - next session: slot 2, how far one update may move: the surrogate objective and PPO, which also uses a critic and GAE
    - then GRPO, a baseline from a group of episodes with no critic
